# NYC yellow taxi trips, January 2024

This notebook explores every yellow taxi trip in New York City in January 2024 with SQL cells, then charts the results with matplotlib.

The data is the SQLite database `examples/nyc_taxi.db`. It is not in the repository: build it once with

```
uv run examples/build_nyc_taxi_db.py
```

It downloads the city's trip records for January 2024 (about 50 MB) and writes about 475 MB, in a few minutes. It has three tables:

| Table | Holds |
| --- | --- |
| `trips` | one row per trip: pickup and dropoff times and zones, distance, fare, tip, tolls and surcharges |
| `zones` | the taxi zones, by `location_id`, with their borough |
| `payment_types` | the names of the `payment_type` codes |

The SQL cells run on a connection called `taxi`. Before running them, add it in **Settings → Data connections**: type SQLite, file `examples/nyc_taxi.db`. Each SQL cell leaves its rows in the dataframe its `--out` names, for the Python cells below to use. See [SQL Cells](https://zasper.io/docs/sql-cells) for more on SQL cells.

## The trips

A first look at the `trips` table. A SQL cell reads at most 1,000 rows unless told otherwise, which is plenty to see the columns. The queries below run on the whole table, in the database, and bring back only their answers.

In [ ]:
%%zasper_sql taxi --out df_trips --limit 1000
SELECT * FROM trips

## Cleaning

Not every row is a ride. Refunds and corrections are recorded as trips with a negative fare, and some trips have no fare or end before they start. The queries below keep only trips with

```sql
fare_amount > 0 AND dropoff_at > pickup_at
```

Trips with a distance of zero stay in: they were paid for.

In [ ]:
%%zasper_sql taxi --out df_dropped
-- Trips the queries below leave out
SELECT sum(fare_amount < 0) AS negative_fare,
       sum(fare_amount = 0) AS zero_fare,
       sum(dropoff_at <= pickup_at) AS ends_before_start,
       sum(fare_amount <= 0 OR dropoff_at <= pickup_at) AS dropped,
       count(*) AS trips
FROM trips

## Trips by borough

Trips, and the average total paid per trip (fare, tip, tolls and surcharges), by the borough of the pickup zone. `Unknown` and `N/A` are zones with no borough.

In [ ]:
%%zasper_sql taxi --out df_boroughs
SELECT z.borough, count(*) AS trips, round(avg(t.total_amount), 2) AS avg_total
FROM trips t JOIN zones z ON z.location_id = t.pickup_location_id
WHERE t.fare_amount > 0 AND t.dropoff_at > t.pickup_at
GROUP BY 1 ORDER BY 2 DESC

## Trips per day

The first ten characters of `pickup_at` are the date, so grouping by them counts the trips on each day.

In [ ]:
%%zasper_sql taxi --out df_daily
SELECT substr(pickup_at, 1, 10) AS day, count(*) AS trips
FROM trips
WHERE fare_amount > 0 AND dropoff_at > pickup_at
GROUP BY 1 ORDER BY 1

## Tips by payment type

All the tips paid in each way, as a percentage of all the fares. Averaging each trip's own tip rate instead would let a few trips with a tiny fare and an ordinary tip, some of them thousands of percent, pull the average up.

In [ ]:
%%zasper_sql taxi --out df_tips
SELECT p.name, count(*) AS trips, round(sum(t.tip_amount) * 100 / sum(t.fare_amount), 1) AS tip_pct
FROM trips t JOIN payment_types p USING (payment_type)
WHERE t.fare_amount > 0 AND t.dropoff_at > t.pickup_at
GROUP BY 1 ORDER BY 2 DESC

## Charts

The first cell sets the colours and the axis style the charts share.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

INK, MUTED, GRID, BLUE, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6", "#fcfcfb"

def style(ax, title):
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.tick_params(colors=MUTED, length=0)
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=13)

def hbars(labels, values, notes, title, tick):
    fig, ax = plt.subplots(figsize=(10, 0.5 * len(labels) + 0.8), facecolor=SURFACE)
    ax.barh(labels, values, height=0.6, color=BLUE)
    for y, (value, note) in enumerate(zip(values, notes)):
        ax.text(value, y, f"  {note}", va="center", color=INK, fontsize=9)
    ax.set_xlim(0, max(values) * 1.35)
    ax.xaxis.set_major_formatter(lambda v, _: tick(v))
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    style(ax, title)
    ax.tick_params(axis="y", colors=INK)
    plt.show()

**Trips per day** follows a weekly rhythm: Wednesdays to Saturdays are the busiest, Sundays and Mondays the quietest. The busiest day of the month and the quietest are marked. Monday, January 15, Martin Luther King Jr. Day, is lower than the Mondays around it.

In [ ]:
daily = df_daily.assign(day=pd.to_datetime(df_daily["day"]))
fig, ax = plt.subplots(figsize=(10, 4), facecolor=SURFACE)
ax.plot(daily["day"], daily["trips"], color=BLUE, linewidth=2)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_ylim(0, daily["trips"].max() * 1.12)
ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f}k" if v else "0")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
for row in (daily.loc[daily["trips"].idxmax()], daily.loc[daily["trips"].idxmin()]):
    ax.scatter(row["day"], row["trips"], s=40, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.annotate(f"{row['trips']:,}", (row["day"], row["trips"]), xytext=(0, 8),
                textcoords="offset points", ha="center", color=INK, fontsize=9)
style(ax, "Yellow taxi trips per day, January 2024")
plt.show()

**Tips by payment type.** Card riders tip 22.5% of the fare. The data records only card tips, so cash trips show no tip whether or not the rider left one.

In [ ]:
tips = df_tips.sort_values("tip_pct")
hbars(tips["name"], tips["tip_pct"],
      [f"{pct:.1f}%  ·  {n:,} trips" for pct, n in zip(tips["tip_pct"], tips["trips"])],
      "Tips as % of fares, by payment type", lambda v: f"{v:.0f}%")

**Average paid per trip, by pickup borough.** Each bar is labelled with its trip count: Staten Island's average rests on 67 trips. EWR is Newark airport.

In [ ]:
# Unknown and N/A are zones with no borough
boroughs = df_boroughs[~df_boroughs["borough"].isin(["Unknown", "N/A"])].sort_values("avg_total")
hbars(boroughs["borough"], boroughs["avg_total"],
      [f"${v:.2f}  ·  {n:,} trips" for v, n in zip(boroughs["avg_total"], boroughs["trips"])],
      "Average paid per trip, by pickup borough", lambda v: f"${v:.0f}")

## Summary

January 2024 had 2.96 million yellow taxi trips. 39,140 of them (1.3%) are refunds, corrections or bad records, and are left out; the figures below are for the other 2.93 million.

- **Manhattan is the taxi market.** 89% of trips start there, and they are the cheapest, at $22.87 on average.
- **Airports make the expensive trips.** Trips from Queens, home to JFK and LaGuardia, average $71.04, three times Manhattan's. The 283 from Newark average $111.49.
- **Demand follows the week.** Between 66,537 trips (Sunday, January 7) and 109,026 (Wednesday, January 17) a day, busiest Wednesday to Saturday, quietest on Sundays, Mondays and Martin Luther King Jr. Day.
- **Card riders tip 22.5% of the fare**, Flex fare riders 7.7%. Cash tips are not recorded, so how much cash riders tip is unknown.

Two things to keep in mind: a few hundred trips from Newark and Staten Island are too few to say much about, and the 140,000 Flex fare trips have no passenger count.